## Matrix of stoichiometric coefficients

### Setup the matrix

\begin{equation}
  \underline{N}^T=
    \begin{bmatrix}
    & CO & CO_2 & CH_3OCH_3 & H_2 & H_2O & CH_3OH \\
R_1 & 0 & -1 & 0 & -3 & 1 & 1 \\
R_2 & -1 & 0 & 0 & -2 & 0 & 1 \\
R_3 & 1 & -1 & 0 & -1 & 1 & 0 \\
R_4 & 0 & 0 & 1 & 0 & 1 & -2
    \end{bmatrix}
\end{equation}

### Python implementation
In the original matrix, the first three reactions ($R_1, R_2, R_3$) were linearly dependent because $R_1 - R_2 = R_3$. Mathematically, this resulted in a Singular Matrix (determinant = 0), making it impossible to calculate the inverse matrix needed for the mass balance. So, $R_4$ was moved to the 3rd column. The rearrangement to $[R_1, R_2, R_4, R_3]$ is a standard engineering requirement to ensure the system of equations is solvable and well-defined.

In [46]:
import numpy as np
from numpy.linalg import matrix_rank

# --- DME Synthesis Stoichiometric Analysis ---
# Component Order: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]
# Reaction Order adjusted: [R1, R2, R4, R3] to ensure N11 is non-singular
# This order ensures Key Components are in the first 3 rows (Rank = 3)

n_matrix = np.array([
    [ 0, -1,  0,  1],  # CO    
    [-1,  0,  0, -1],  # CO2   
    [ 0,  0,  1,  0],  # CH3OCH3 (DME) -> Now independent in the 3rd column (because R1-R2=R3)
    [-3, -2,  0, -1],  # H2    
    [ 1,  0,  1,  1],  # H2O   
    [ 1,  1, -2,  0]   # CH3OH
])

# Get the transposed matrix (N^T)
n_transposed = n_matrix.T

# Determine the rank of the matrix
n_rank = matrix_rank(n_matrix)

print("--- DME Synthesis Stoichiometric Analysis ---")
print(f"Sequence: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]")
print("\nTransposed Matrix (N^T):")
print(n_transposed)
print(f"\nMatrix Rank (Number of Key Reactions): {n_rank}")

--- DME Synthesis Stoichiometric Analysis ---
Sequence: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]

Transposed Matrix (N^T):
[[ 0 -1  0 -3  1  1]
 [-1  0  0 -2  0  1]
 [ 0  0  1  0  1 -2]
 [ 1 -1  0 -1  1  0]]

Matrix Rank (Number of Key Reactions): 3


The rank is $R_N=3$, which means that two key reactions and components are sufficient to describe the reaction extent based on stoichiometry. According to the order of components and reactions chosen for the matrix of stoichiometric coefficients, $CO$, $CO_2$, $CH_3OCH_3 (DME)$ are the key components. 

## Determination of the reaction extent

### Material balance

In [47]:
# Partitioning the stoichiometric matrix for DME Synthesis
# Based on the calculated rank (n_rank = 3)

# N11: Square sub-matrix representing key components in key reactions (3x3)
# With the updated n_matrix, this is now non-singular (invertible)
n_matrix_11 = n_matrix[0:n_rank, 0:n_rank]

# N21: Remaining components in key reactions
n_matrix_21 = n_matrix[n_rank:n_matrix.shape[0], 0:n_rank]

# N12: Key components in dependent reactions (if any)
n_matrix_12 = n_matrix[0:n_rank, n_rank:n_matrix.shape[1]]

# N22: Remaining components in dependent reactions
n_matrix_22 = n_matrix[n_rank:n_matrix.shape[0], n_rank:n_matrix.shape[1]]

print("--- Matrix Partitioning Completed ---")
print(f"Dimension of N11 (should be square): {n_matrix_11.shape}")

--- Matrix Partitioning Completed ---
Dimension of N11 (should be square): (3, 3)


### Conversion and flow rates

assume:
\begin{equation}
  \underline{\dot n}_{in}=
    \begin{bmatrix}
      30.0 & CO\\ 10.0 & CO_2\\ 0.0 & CH_3OCH_3\\ 80.0 & H_2\\ 0.0 & H_2O\\ 0.0 & CH_3OH\\
    \end{bmatrix}
    \frac{mol}{s}
\end{equation}

\begin{equation}
  \Delta\underline{\dot n}_{1}=
    \begin{bmatrix}
      -24.0 & CO\\ -4.0 & CO_2\\ 12.0 & CH_3OCH_3
    \end{bmatrix}
    \frac{mol}{s}
\end{equation}

In [48]:
# --- Conversion and Flow Rates Calculation ---
# Sequence: [CO, CO2, DME, H2, H2O, MeOH]

n_in = np.array([30.0, 10.0, 0.0, 80.0, 0.0, 0.0]) # inlet molar flow rates in mol/s
Delta_n1 = np.array([-24.0, -4.0, 12.0]) # measured conversion of key components in mol/s

# Formula: Delta_n2 = N21 * inv(N11) * Delta_n1
# This calculates the conversion of non-key components (H2, H2O, MeOH)
Delta_n2 = np.matmul(np.matmul(n_matrix_21, np.linalg.inv(n_matrix_11)), Delta_n1) 
print('conversion of non-key components in mol/s:', Delta_n2) # display key result

# Calculating final outlet molar flow rates
n_out = n_in + np.hstack((Delta_n1, Delta_n2)) 
print('outlet molar flow rates in mol/s:', n_out) # display key result

conversion of non-key components in mol/s: [-60.  16.   4.]
outlet molar flow rates in mol/s: [ 6.  6. 12. 20. 16.  4.]


The conversion of the non-key components is now known:
\begin{equation}
  \Delta\underline{\dot n}_{2}=
    \begin{bmatrix}
      -60.0 & H_2\\ -16.0 & H_2O \\ 4.0 & CH_3OH
    \end{bmatrix}
    \frac{mol}{s}
\end{equation}

The outlet molar flow rates of all components are:
\begin{equation}
  \underline{\dot n}_{out}=
    \begin{bmatrix}
      6.0 & CO\\ 6.0 & CO_2\\ 12.0 & CH_3OCH_3\\ 20.0 & H2\\ 16.0 & H_2O \\ 4.0 & CH_3OH
    \end{bmatrix}
    \frac{mol}{s}
\end{equation}


### Reaction extent

In [49]:
# solving linear equation system for reaction extents
rxn_ext = np.linalg.solve(n_matrix_11, Delta_n1) 

# display of key result
print('reaction extent in mol/s:', rxn_ext)

reaction extent in mol/s: [ 4. 24. 12.]


The reaction extent of the key reactions, thus, amounts to:

\begin{equation}
  \underline{\dot \xi}=
    \begin{bmatrix}
      4.0 \\ 24.0 \\ 12.0
    \end{bmatrix}
    \frac{mol}{s}
\end{equation}

$\dot{\xi}_1 = 4.0 \text{ mol/s}$: $CO_2$ 

$\dot{\xi}_2 = 24.0 \text{ mol/s}$: $CO$ 

$\dot{\xi}_3 = 12.0 \text{ mol/s}$: $CH_3OCH_3$